# ⚡ Interactive Electric Field Simulator

This simulation visualizes electric field lines between two point charges.

- 🔴 **Red** = positive charge
- 🔵 **Blue** = negative charge
- ✅ **Probe** = measures field strength and direction at any point

Use the sliders to move the charges, change their strength, and explore the field!

▶️ **Run All cells** from the Runtime menu to start.

In [ ]:
!pip install numpy matplotlib ipywidgets -q

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from ipywidgets import FloatSlider, VBox, HBox, Label
from IPython.display import display

GRID_N = 80
grid_range = 3.0
x, y = np.meshgrid(np.linspace(-grid_range, grid_range, GRID_N),
                   np.linspace(-grid_range, grid_range, GRID_N))

def compute_field(charges):
    Ex, Ey = np.zeros_like(x), np.zeros_like(y)
    for cx, cy, q in charges:
        dx, dy = x - cx, y - cy
        r = np.sqrt(dx**2 + dy**2)
        r = np.maximum(r, 0.05)
        Ex += q * dx / r**3
        Ey += q * dy / r**3
    return Ex, Ey

def field_at_point(charges, px, py):
    Ex, Ey = 0.0, 0.0
    for cx, cy, q in charges:
        dx, dy = px - cx, py - cy
        r = np.sqrt(dx**2 + dy**2)
        r = max(r, 0.05)
        Ex += q * dx / r**3
        Ey += q * dy / r**3
    return Ex, Ey

def plot_field(q1x, q1y, q1, q2x, q2y, q2, px, py):
    charges = [[q1x, q1y, q1], [q2x, q2y, q2]]
    Ex, Ey = compute_field(charges)
    magnitude = np.sqrt(Ex**2 + Ey**2)
    log_mag = np.log1p(magnitude)

    fig = plt.figure(figsize=(14, 6), facecolor='#0d0d1a')
    gs = gridspec.GridSpec(1, 3, figure=fig, left=0.05, right=0.97,
                           top=0.90, bottom=0.10, wspace=0.35)
    ax_main = fig.add_subplot(gs[0, 0:2])
    ax_info  = fig.add_subplot(gs[0, 2])

    for ax in [ax_main, ax_info]:
        ax.set_facecolor('#0d0d1a')

    ax_main.streamplot(x, y, Ex, Ey,
                       color=log_mag, cmap='plasma',
                       density=2.0, linewidth=0.8, arrowsize=1.2)

    for cx, cy, q in charges:
        color = '#ff4444' if q >= 0 else '#4488ff'
        label = f'+{q:.1f}' if q >= 0 else f'{q:.1f}'
        circle = plt.Circle((cx, cy), 0.12, color=color, zorder=5)
        ax_main.add_patch(circle)
        ax_main.text(cx, cy + 0.25, label, color=color,
                     ha='center', fontsize=10, fontweight='bold', zorder=6)

    ax_main.plot(px, py, 'X', color='#00ffaa', markersize=12, zorder=7)
    ax_main.text(px + 0.1, py + 0.25, 'probe', color='#00ffaa', fontsize=9, zorder=8)
    ax_main.set_xlim(-grid_range, grid_range)
    ax_main.set_ylim(-grid_range, grid_range)
    ax_main.set_title('Electric Field Lines', color='#e0e0ff', fontsize=13, pad=10)
    ax_main.tick_params(colors='#555577')
    ax_main.set_xlabel('x', color='#7070aa')
    ax_main.set_ylabel('y', color='#7070aa')
    for spine in ax_main.spines.values():
        spine.set_edgecolor('#333355')

    Epx, Epy = field_at_point(charges, px, py)
    E_mag   = np.sqrt(Epx**2 + Epy**2)
    E_angle = np.degrees(np.arctan2(Epy, Epx))

    ax_info.axis('off')
    ax_info.set_title('Field at Probe', color='#e0e0ff', fontsize=12, pad=10)
    lines = [
        ('Position',  f'({px:.2f},  {py:.2f})'),
        ('Ex',        f'{Epx:+.3f}'),
        ('Ey',        f'{Epy:+.3f}'),
        ('|E|',       f'{E_mag:.3f}'),
        ('Angle',     f'{E_angle:.1f}°'),
    ]
    for i, (label, value) in enumerate(lines):
        ax_info.text(0.05, 0.82 - i*0.15, label + ':', color='#7070cc',
                     fontsize=12, transform=ax_info.transAxes)
        ax_info.text(0.55, 0.82 - i*0.15, value, color='#00ffaa',
                     fontsize=12, fontweight='bold', transform=ax_info.transAxes)

    norm = E_mag if E_mag > 0 else 1
    ux, uy = Epx / norm, Epy / norm
    ax_info.annotate('', xy=(0.5 + ux*0.25, 0.15 + uy*0.15),
                     xytext=(0.5, 0.15), xycoords='axes fraction',
                     arrowprops=dict(arrowstyle='->', color='#00ffaa',
                                     lw=2.5, mutation_scale=20))
    ax_info.text(0.5, 0.04, 'field direction', color='#7070cc',
                 fontsize=9, ha='center', transform=ax_info.transAxes)

    fig.suptitle('⚡ Interactive Electric Field Simulator',
                 color='#e0e0ff', fontsize=14, fontweight='bold')
    plt.show()

slider_style  = {'description_width': '80px'}
slider_layout = widgets.Layout(width='320px')

q1x = FloatSlider(value=-1.0, min=-2.5, max=2.5, step=0.1, description='Q1  x',     style=slider_style, layout=slider_layout)
q1y = FloatSlider(value=0.0,  min=-2.5, max=2.5, step=0.1, description='Q1  y',     style=slider_style, layout=slider_layout)
q1  = FloatSlider(value=1.0,  min=-3.0, max=3.0, step=0.1, description='Q1 charge', style=slider_style, layout=slider_layout)
q2x = FloatSlider(value=1.0,  min=-2.5, max=2.5, step=0.1, description='Q2  x',     style=slider_style, layout=slider_layout)
q2y = FloatSlider(value=0.0,  min=-2.5, max=2.5, step=0.1, description='Q2  y',     style=slider_style, layout=slider_layout)
q2  = FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.1, description='Q2 charge', style=slider_style, layout=slider_layout)
px  = FloatSlider(value=0.0,  min=-2.5, max=2.5, step=0.1, description='Probe x',   style=slider_style, layout=slider_layout)
py  = FloatSlider(value=0.0,  min=-2.5, max=2.5, step=0.1, description='Probe y',   style=slider_style, layout=slider_layout)

col1 = VBox([Label('── Charge 1 ──'), q1x, q1y, q1])
col2 = VBox([Label('── Charge 2 ──'), q2x, q2y, q2])
col3 = VBox([Label('── Probe Point ──'), px, py])
controls = HBox([col1, col2, col3])

out = widgets.interactive_output(plot_field, {
    'q1x': q1x, 'q1y': q1y, 'q1': q1,
    'q2x': q2x, 'q2y': q2y, 'q2': q2,
    'px': px, 'py': py
})

display(controls, out)